<a href="https://colab.research.google.com/github/sebassanchez5680-tech/Prueba/blob/main/Sesion10_Data_Profiling_Entregable_271757.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 10 — Entregable sobre Data Profiling

**Nombre completo: Sebastian Miguel Sanchez Salas**

**Matrícula:271757**

---

Este notebook contiene las actividades a entregar de la Sesión 10 (Data Profiling), aplicadas sobre **datasets reales**: el catálogo de Netflix y el dataset de Customer Personality Analysis (Kaggle). Si aún no revisaste las explicaciones y ejemplos de cada tema, hazlo primero en el notebook `Sesion10_Data_Profiling_Actividad_Asincrona.ipynb`.

**Nota sobre los datos:** ambos datasets son reales — los problemas de calidad que vas a encontrar (nulos, categorías inconsistentes, valores fuera de rango) ya existían antes de que este notebook los usara. La única excepción está marcada explícitamente en la Actividad 3 y la Práctica integradora, donde se inyectan un par de filas/valores a propósito para poder practicar duplicados y ajuste de tipos con un resultado garantizado.

**Antes de entregar:** ejecuta "Reiniciar y ejecutar todo" para confirmar que tu notebook corre de principio a fin sin errores.

## Preparación

Ejecuta esta celda antes de empezar — descarga los dos datasets reales que vas a usar.

In [98]:
import pandas as pd

url_netflix = 'https://raw.githubusercontent.com/Vibe1990/Netflix-Project/main/netflix_title.csv'
url_marketing = 'https://raw.githubusercontent.com/amankharwal/Website-data/master/marketing_campaign.csv'

df_netflix = pd.read_csv(url_netflix)
df_marketing = pd.read_csv(url_marketing, sep=';')  # nota: este archivo usa punto y coma, no coma

print('Netflix:', df_netflix.shape)
print('Marketing:', df_marketing.shape)

Netflix: (7787, 12)
Marketing: (2240, 29)


---
## Actividad 1 — Renombrado y estandarización de columnas

*Dataset: Customer Personality Analysis*

Revisa los nombres de columna de `df_marketing` con `.columns`. Vas a notar una mezcla de convenciones reales: `Year_Birth` (con guion bajo), `Kidhome` (sin separador), `MntWines` (abreviado y sin separador).

**Trabaja sobre una copia** (`df_marketing_renombrado = df_marketing.copy()`) para no afectar las actividades siguientes, que usan los nombres originales. Aplica `.str.lower()` para al menos unificar mayúsculas/minúsculas, y usa `.rename()` para corregir manualmente los 2-3 nombres que la técnica automática no deja perfectos (por ejemplo, `mntwines` sigue sin ser ideal — decide tú el nombre final).

In [99]:
print(df_marketing.columns) #visualizamos las columnas del df original

Index(['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome',
       'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits',
       'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
       'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
       'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth',
       'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1',
       'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response'],
      dtype='object')


In [100]:

df_marketing_renombrado = df_marketing.copy() # creamos una copia del dataframe original, que es el cual vamos a trabajar para mantener el original intacto

df_marketing_renombrado.columns = df_marketing_renombrado.columns.str.lower()# usamos .str.lower para dejar todo en minusculas

df_marketing_renombrado = df_marketing_renombrado.rename(columns={'kidhome': 'kid_home','mntwines': 'mnt_wines','teenhome': 'teen_home'}) #corregimos nombre con  .rename, agregamos guiones bajos para separar

print(df_marketing_renombrado.columns)

Index(['id', 'year_birth', 'education', 'marital_status', 'income', 'kid_home',
       'teen_home', 'dt_customer', 'recency', 'mnt_wines', 'mntfruits',
       'mntmeatproducts', 'mntfishproducts', 'mntsweetproducts',
       'mntgoldprods', 'numdealspurchases', 'numwebpurchases',
       'numcatalogpurchases', 'numstorepurchases', 'numwebvisitsmonth',
       'acceptedcmp3', 'acceptedcmp4', 'acceptedcmp5', 'acceptedcmp1',
       'acceptedcmp2', 'complain', 'z_costcontact', 'z_revenue', 'response'],
      dtype='object')


---
## Actividad 2 — Ajuste de tipos: fechas con formato mixto

*Dataset: Netflix*

La columna `date_added` de `df_netflix` mezcla formatos reales: la mayoría son `"14-Aug-20"`, pero un grupo minoritario llega como `" August 4, 2017"` (con espacio inicial). Conviértela a tipo fecha usando `pd.to_datetime(..., format='mixed')`, que resuelve ambos formatos en la misma columna. Verifica con `.dtypes` y confirma cuántos valores nulos quedan después de la conversión (compara contra los nulos que ya traía antes de convertir).

In [101]:
df_netflix['date_added'] = pd.to_datetime(df_netflix['date_added'], format='mixed') # con to_datetime convertimos la columna de texto a tipo de fecha

print("Tipo de dato :",df_netflix.dtypes['date_added']) #con dtypes verificamos el tipo de dato de la columna 'date added'

print("Valores nulos:", df_netflix['date_added'].isnull().sum()) #finalmente hacemos el conteo de los valores que no pudieron cambiar su tipo

Tipo de dato : datetime64[ns]
Valores nulos: 10


---
## Actividad 3 — Duplicados

*Dataset: Customer Personality Analysis — con 2 filas duplicadas inyectadas a propósito*

Este dataset real no trae duplicados de forma natural — para poder practicar, se insertan 2 copias de clientes ya existentes (ejecuta la celda siguiente).

In [102]:
df_marketing_dup = pd.concat([df_marketing, df_marketing.sample(2, random_state=7)], ignore_index=True)
print('Filas originales:', len(df_marketing))
print('Filas con duplicados inyectados:', len(df_marketing_dup))

Filas originales: 2240
Filas con duplicados inyectados: 2242


Sobre `df_marketing_dup`: cuenta los duplicados exactos con `.duplicated().sum()`, luego cuenta los duplicados por `ID` con `.duplicated(subset='ID').sum()` (deberían coincidir, ya que el `ID` es único por cliente). Elimínalos con `.drop_duplicates()` y confirma el número final de filas.

In [103]:

print("Duplicados exactos:", df_marketing_dup.duplicated().sum()) #contamos los duplicados de todas las filas
print("Duplicados por ID:", df_marketing_dup.duplicated(subset='ID').sum())#de igual manera contamos los datos duplicados pero tomando en cuenta al id de los clientes

df_marketing_dup = df_marketing_dup.drop_duplicates(subset='ID', keep='first') #usamos drop_duplicates para eliminar los duplicados agregados
print("Filas tras limpiar:", len(df_marketing_dup)) #validamos que efectivamente los duplicados se hayan eliminado

Duplicados exactos: 2
Duplicados por ID: 2
Filas tras limpiar: 2240


---
## Actividad 4 — Valores faltantes

*Dataset: Netflix*

Usa `.isnull().sum()` sobre `df_netflix` para ver cuántos valores faltan por columna. Luego, usa `.isnull().any(axis=1)` para filtrar solo las filas que tienen **al menos un** valor faltante en cualquier columna, y muestra cuántas filas son en total (`.sum()` sobre el resultado booleano).

In [104]:
print("Nulos por columna :", df_netflix.isnull().sum()) #hacemos un recuento de los valores nulos por columnas de doto el data set

print("\nFilas totales con al menos un nulo:", df_netflix.isnull().any(axis=1).sum()) # de igual manera hacemos un recuento de los valores nulos pero esta vez de las lineas

Nulos por columna : show_id            0
type               0
title              0
director        2389
cast             718
country          507
date_added        10
release_year       0
rating             7
duration           0
listed_in          0
description        0
dtype: int64

Filas totales con al menos un nulo: 2979


---
## Actividad 5 — Completitud como porcentaje

*Dataset: Netflix*

Con los mismos nulos de la Actividad 4, calcula la completitud en porcentaje por columna: `(1 - nulos / total_filas) * 100`. ¿Qué columna tiene la completitud más baja? Escribe la respuesta en una línea.

In [105]:
total_netflix = len(df_netflix) #el total de registros de netflix
completitud_netflix = (1 - df_netflix.isnull().sum() / total_netflix) * 100 #obtenemos el porcentaje

print(completitud_netflix)

# la columna con la completitud mas baja es la de director que como vimos en la actividad 4, es la columna con mas valores nulos

show_id         100.000000
type            100.000000
title           100.000000
director         69.320663
cast             90.779504
country          93.489149
date_added       99.871581
release_year    100.000000
rating           99.910107
duration        100.000000
listed_in       100.000000
description     100.000000
dtype: float64


---
## Actividad 6 — Exploración categórica

*Dataset: Customer Personality Analysis*

Aplica `.value_counts()` sobre la columna `Marital_Status` de `df_marketing`. Vas a encontrar, junto a las categorías esperadas (`Married`, `Single`, `Together`, `Divorced`, `Widow`), tres valores que claramente son errores de captura reales: `Alone`, `Absurd` y `YOLO`. Decide y justifica en una línea: ¿los eliminarías, los reclasificarías (por ejemplo, `Alone` → `Single`), o los dejarías así? No hay una única respuesta correcta — lo que importa es la justificación.

In [106]:
print(df_marketing['Marital_Status'].value_counts()) #value_counts nos permitira visualizar los valores con los que cuenta cada categoria

#alone lo reclasificaria ya que claramente ya hay una categoria en la cual pertence
# mientras que absurd y yolo son digamos, valores atipicos, los cual los eliminaremos ya que no aportan valor a la base

Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64


---
## Actividad 7 — Consistencia de formato/patrón

*Dataset: Netflix*

La columna `show_id` debería seguir siempre el patrón: la letra `s` seguida de uno o más dígitos (`s1`, `s2`, ..., `s8807`). Verifica con `.str.match(r'^s\d+$')` si todos los valores cumplen esta convención. Reporta el porcentaje de cumplimiento.

In [107]:
patron = df_netflix['show_id'].str.match(r'^s\d+$') # con str.match comprobamos si el inicio de cada texto coincide con el patron requerido

porcentaje_cumplimiento = patron.mean()* 100 #calculamos el cumplimiento
print(f"Cumplimiento del patrón: {porcentaje_cumplimiento:.2f}%")

Cumplimiento del patrón: 100.00%


---
## Actividad 8 — `.info()` y `.describe()`

*Dataset: Customer Personality Analysis*

Ejecuta `.describe()` sobre la columna `Year_Birth` de `df_marketing` (puedes hacerlo con `df_marketing[['Year_Birth']].describe()`). Observa el valor mínimo (`min`). ¿Tiene sentido ese año de nacimiento? Filtra el DataFrame para mostrar las filas con los años de nacimiento más antiguos y decide, en una línea, si los considerarías un error de captura.

In [108]:
print(df_marketing[['Year_Birth']].describe())

print(df_marketing[df_marketing['Year_Birth'] < 1930][['ID', 'Year_Birth']]) #filtrar los registros mas viejos mas viejos

#los registros de estos clientes tienen años de nacimiento muy antiguos puede ser que se trate de clientes muy viejos, los descartaria ya que son valores atipicos

        Year_Birth
count  2240.000000
mean   1968.805804
std      11.984069
min    1893.000000
25%    1959.000000
50%    1970.000000
75%    1977.000000
max    1996.000000
        ID  Year_Birth
192   7829        1900
239  11004        1893
339   1150        1899


---
## Actividad 9 — Práctica integradora: checklist de profiling

*Dataset: muestra real de Customer Personality Analysis, con 2 elementos inyectados y marcados a propósito (una fila duplicada y un valor de tipo incorrecto) para poder practicar el checklist completo con un resultado garantizado.*

Aplica el checklist completo, en orden, sobre `df_practica`:

1. Revisa `.dtypes` e identifica qué columna tiene un problema de tipo, corrígela con `pd.to_numeric(errors='coerce')`
2. Cuenta las filas duplicadas y elimínalas con `.drop_duplicates()`
3. Cuenta los valores faltantes por columna con `.isnull().sum()` (incluyendo el que se generó en el paso 1)
4. Revisa `.unique()` sobre `Marital_Status` y decide si necesita normalización

Al final, escribe un breve "reporte de profiling" (3-4 líneas) resumiendo qué encontraste y qué decidiste.

In [109]:
# Muestra real con 2 elementos inyectados (marcados abajo)
df_practica = df_marketing.sample(15, random_state=3).reset_index(drop=True).copy()

# Elemento inyectado 1: una fila duplicada
df_practica = pd.concat([df_practica, df_practica.iloc[[2]]], ignore_index=True)

# Elemento inyectado 2: un valor de tipo incorrecto en Income
df_practica['Income'] = df_practica['Income'].astype(object)
df_practica.loc[5, 'Income'] = 'sesenta mil'

df_practica[['ID', 'Marital_Status', 'Income']]

,ID,Marital_Status,Income
0,5788,Together,46053.0
1,7930,Single,26877.0
2,4557,Together,22070.0
3,9964,Single,61825.0
4,1168,Married,72159.0
5,5314,Together,sesenta mil
6,9665,Divorced,54237.0
7,6182,Together,26646.0
8,922,Married,31086.0
9,4427,Single,83257.0


In [110]:
df_practica['Income'] = pd.to_numeric(df_practica['Income'], errors='coerce') # ajustamos la columna para transformar los datos a que sean numericos
print(df_practica['Income']) #visualizacion de como se ve la nueva columna ajustada

0     46053.0
1     26877.0
2     22070.0
3     61825.0
4     72159.0
5         NaN
6     54237.0
7     26646.0
8     31086.0
9     83257.0
10    76005.0
11    70643.0
12    70091.0
13    50898.0
14    65316.0
15    22070.0
Name: Income, dtype: float64


In [111]:
print("Duplicados :", df_practica.duplicated().sum()) #existe algun duplicado?
df_practica = df_practica.drop_duplicates() #eliminamos los duplicados
print("Duplicados despues de la eliminacion:", df_practica.duplicated().sum()) #existe algun duplicado?

Duplicados : 1
Duplicados despues de la eliminacion: 0


In [112]:
print("Nulos detectados:\n", df_practica.isnull().sum()) # Conteo de nulos actualizados

Nulos detectados:
 ID                     0
Year_Birth             0
Education              0
Marital_Status         0
Income                 1
Kidhome                0
Teenhome               0
Dt_Customer            0
Recency                0
MntWines               0
MntFruits              0
MntMeatProducts        0
MntFishProducts        0
MntSweetProducts       0
MntGoldProds           0
NumDealsPurchases      0
NumWebPurchases        0
NumCatalogPurchases    0
NumStorePurchases      0
NumWebVisitsMonth      0
AcceptedCmp3           0
AcceptedCmp4           0
AcceptedCmp5           0
AcceptedCmp1           0
AcceptedCmp2           0
Complain               0
Z_CostContact          0
Z_Revenue              0
Response               0
dtype: int64


In [113]:
print("\nCategorías en Marital_Status:", df_practica['Marital_Status'].unique()) #que categorias hay


Categorías en Marital_Status: ['Together' 'Single' 'Married' 'Divorced']


**Tu reporte de profiling:**

*"El analisis revelo que la columna Income contenia texto ('sesenta mil'), el cual fue forzado a hacerse un NaN a la hora de ajustarlo a numerico. Se identificó y elimino una fila duplicada, y la variable Marital_Status mantiene sus categorías estandarizadas sin errores de escritura en esta muestra. Los valores nulos están ahora concentrados en los registros donde las conversiones fallaron o faltaban desde el origen."*